# QuickTalk: train a small model from scratch (Colab, A100)

Trains a GPT-style model on `train/build/` from the `sraivante/quicktalk` repo: first on plain text (books + behaviour
passages), then on chats (behaviour Q&A + 12 conversation patterns).

**Before you start**
1. *Runtime > Change runtime type > A100 GPU*. The notebook stops if the GPU is not an A100.
2. The repo is public, so no token is needed. (Only if it is made private later: add a Colab secret named
   `GITHUB_TOKEN` with a GitHub token that can read the repo.)
3. *Runtime > Run all*.

**Crashes, disconnects and idle timeouts**
- Everything is saved to Google Drive (`MyDrive/quicktalk_run`): tokenizer, token files, checkpoints every 5 minutes,
  logs. If the runtime is killed, reconnect and *Run all* again: finished steps are skipped and training resumes from
  the last checkpoint.
- Training runs in the background, so closing the browser tab does not stop it while the runtime is alive. With Colab
  Pro+ you can also turn on background execution so the runtime keeps running after you close the tab.
- The whole run is short (see the plan cell for the step count), so a session timeout is unlikely to hit it.

**Model size:** the plan cell uses your rule of 10 training tokens per parameter (not Chinchilla's 20).

In [ ]:
# 1. Check the GPU: A100 only
import subprocess, torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > A100 GPU'
name = torch.cuda.get_device_name(0)
print('GPU:', name)
assert 'A100' in name, f'This runtime has {name}, not an A100. Change the runtime type to A100 and run again.'

In [ ]:
# 2. Google Drive holds all state, so nothing is lost if the runtime restarts
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/quicktalk_run'
os.makedirs(WORK, exist_ok=True)
print('work dir:', WORK, os.listdir(WORK))

In [ ]:
# 3. Get the code and the built data (sparse clone: only train/build and train/colab)
import os, subprocess
REPO, BRANCH, SRC = 'sraivante/quicktalk', 'claude/intelligent-ride-oxqjfy', '/content/quicktalk'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists(f'{SRC}/train/build/manifest.json'):
    subprocess.run(['rm', '-rf', SRC])
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--filter=blob:none', '--sparse', url, SRC], check=True)
    subprocess.run(['git', '-C', SRC, 'sparse-checkout', 'set', 'train/build', 'train/colab'], check=True)
print(sorted(os.listdir(f'{SRC}/train/build')))
LM = f'{SRC}/train/colab/quicktalk_lm.py'
DATA = f'{SRC}/train/build'

In [ ]:
# 4. Libraries
!pip -q install tokenizers

In [ ]:
# 5. Tokenizer, token files and model-size plan (each step skips itself if already done in Drive)
!python {LM} tokenizer --data {DATA} --workdir {WORK}
!python {LM} prepare   --data {DATA} --workdir {WORK}
# 10 tokens per parameter; 4 passes over the pretraining text, 2 over the chats. Change the numbers here if you like.
!python {LM} plan --workdir {WORK} --tokens-per-param 10 --pretrain-epochs 4 --sft-epochs 2

In [ ]:
# 6. Start training in the background (pretrain, then chat fine-tuning). Safe to re-run: it resumes.
import os, subprocess
os.makedirs('/content/cache', exist_ok=True)
subprocess.run(f'rsync -a {WORK}/data/ /content/cache/data/', shell=True, check=True)   # local copy reads faster than Drive
LOG = f'{WORK}/train.log'
cmd = (f'python {LM} train --workdir {WORK} --data-dir /content/cache/data --stage pretrain && '
       f'python {LM} train --workdir {WORK} --data-dir /content/cache/data --stage sft')
running = subprocess.run("pgrep -f '[q]uicktalk_lm.py train'", shell=True, capture_output=True).stdout.strip()
if running:
    print('training is already running; watch it in the next cell')
else:
    subprocess.Popen(f'nohup bash -c "{cmd}" >> {LOG} 2>&1 &', shell=True)
    print('started; log:', LOG)

In [ ]:
# 7. Watch progress (re-run this cell any time; stop it with the stop button, training keeps going)
import time, subprocess, os
LOG = f'{WORK}/train.log'
while True:
    out = subprocess.run(f'tail -n 6 {LOG}', shell=True, capture_output=True, text=True).stdout
    print(out, flush=True)
    done = os.path.exists(f'{WORK}/checkpoints/sft_final.pt')
    alive = subprocess.run("pgrep -f '[q]uicktalk_lm.py train'", shell=True, capture_output=True).stdout.strip()
    if done: print('DONE: model at', f'{WORK}/checkpoints/sft_final.pt'); break
    if not alive: print('training is not running: check the log above, then re-run cell 6 to resume'); break
    time.sleep(60)

In [ ]:
# 8. Loss curves (step, train loss, eval loss)
import pandas as pd, matplotlib.pyplot as plt, os
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for i, st in enumerate(['pretrain', 'sft']):
    p = f'{WORK}/log_{st}.csv'
    if os.path.exists(p):
        d = pd.read_csv(p, names=['step', 'train', 'eval'])
        ax[i].plot(d.step, d.train, label='train'); ax[i].plot(d.step, d.eval, label='eval'); ax[i].set_title(st); ax[i].legend()
plt.show()

In [ ]:
# 9. Talk to the model
!python {LM} chat --workdir {WORK} --prompt "Hi! How was your day?" --prompt "What should I do if my friend is upset with me?" --prompt "Correct this: She dont like apples."